<a href="https://colab.research.google.com/github/SOARESRafaelAlmeida/Sistema-de-Gerenciamento-de-Estacionamento-Inteligente/blob/main/KB_File_Hunter_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knowledge-Based (KB) File Hunter
## Lógica Proposicional, Lógica de Primeira Ordem e Poda Lógica

**Objetivo:** mapear os metadados do sistema de arquivos, construir uma Base de Conhecimento com `TELL/ASK`, inferir candidatos prioritários e comparar BFS, A* tradicional e um agente lógico.

## 1. Preparar o ambiente e localizar a pasta `Fontes`

In [ ]:
# No Google Colab, esta célula permite enviar o ZIP do professor.
# Se o ZIP já estiver extraído em /content/arquivo_professor, pode pular o upload.
import os
from pathlib import Path
import zipfile
import time
import tracemalloc
import heapq
import itertools
import re
import unicodedata
import pandas as pd

RAIZ_CANDIDATA = Path("/content/arquivo_professor")

if not RAIZ_CANDIDATA.exists():
    try:
        from google.colab import files
        print("Envie o ZIP disponibilizado pelo professor.")
        enviados = files.upload()
        for nome in enviados:
            if nome.lower().endswith(".zip"):
                destino = Path("/content/arquivo_professor")
                destino.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(nome, "r") as zf:
                    zf.extractall(destino)
                print(f"Extraído: {nome} -> {destino}")
    except ImportError:
        print("Ambiente fora do Colab. Coloque os arquivos em uma pasta acessível e ajuste a busca abaixo.")

# Procura uma pasta chamada Fontes em locais comuns e em /content.
possiveis = [
    Path("/content/arquivo_professor/Fontes"),
    Path("/content/Fontes"),
    Path("./docencia/fontes"),
    Path("./Fontes"),
]
RAIZ = next((p.resolve() for p in possiveis if p.exists() and p.is_dir()), None)

if RAIZ is None:
    bases = [Path("/content/arquivo_professor"), Path("/content"), Path(".")]
    for base in bases:
        if base.exists():
            encontrados = [p for p in base.rglob("*") if p.is_dir() and p.name.lower() == "fontes"]
            if encontrados:
                RAIZ = encontrados[0].resolve()
                break

if RAIZ is None:
    raise FileNotFoundError("Não encontrei uma pasta chamada 'Fontes'. Verifique se o ZIP foi extraído.")

print("Pasta-raiz utilizada:", RAIZ)
print("Pastas iniciais:", [p.name for p in RAIZ.iterdir() if p.is_dir()][:15])

Envie o ZIP disponibilizado pelo professor.


Saving Fontes-20261019T170335Z-1-001.zip to Fontes-20261019T170335Z-1-001.zip
Extraído: Fontes-20261019T170335Z-1-001.zip -> /content/arquivo_professor
Pasta-raiz utilizada: /content/arquivo_professor/Fontes
Pastas iniciais: ['Fontes GTI', 'Fontes BD', 'Fontes GD', 'Fontes MD', 'Fontes DMN', 'Fontes PDS', 'Fontes PAA', 'Fontes UDP', 'Fontes PCD']


## 2. Motor KB: `TELL` e `ASK`

In [ ]:
# Motor de inferência didático com fatos e regras Horn.
# Sintaxe de fatos: Predicado(termo1, termo2)
# Sintaxe de regras: Cabeca(X) :- Corpo1(X), Corpo2(X, "valor")
# Variáveis são identificadores iniciados por letra maiúscula, como X, Y.

def dividir_argumentos(texto):
    args, atual, aspas, nivel = [], "", False, 0
    for ch in texto:
        if ch == '"':
            aspas = not aspas
            atual += ch
        elif not aspas and ch == "(":
            nivel += 1
            atual += ch
        elif not aspas and ch == ")":
            nivel -= 1
            atual += ch
        elif not aspas and nivel == 0 and ch == ",":
            args.append(atual.strip())
            atual = ""
        else:
            atual += ch
    if atual.strip():
        args.append(atual.strip())
    return args

def parse_atom(texto):
    texto = texto.strip()
    m = re.fullmatch(r"([A-Za-z_]\w*)\((.*)\)", texto)

    if not m:
        raise ValueError(f"Predicado inválido: {texto}")

    nome, corpo = m.group(1), m.group(2).strip()

    if not corpo:
        return nome, tuple()

    argumentos = dividir_argumentos(corpo)

    # Remove as aspas externas dos valores literais.
    argumentos = [
        arg[1:-1] if len(arg) >= 2
        and arg.startswith('"') and arg.endswith('"')
        else arg
        for arg in argumentos
    ]

    return nome, tuple(argumentos)

def eh_variavel(t):
    return bool(re.fullmatch(r"[A-Z][A-Za-z_0-9]*", t.strip()))

def valor_termo(t, subst):
    t = t.strip()
    if t.startswith('"') and t.endswith('"'):
        return t[1:-1]
    if eh_variavel(t):
        return subst.get(t, t)
    return t

def unificar_argumentos(padroes, valores, subst):
    if len(padroes) != len(valores):
        return None
    nova = dict(subst)
    for p, v in zip(padroes, valores):
        p = p.strip()
        if eh_variavel(p):
            if p in nova and nova[p] != v:
                return None
            nova[p] = v
        else:
            if valor_termo(p, nova) != v:
                return None
    return nova

class KnowledgeBase:
    def __init__(self):
        self.fatos = set()
        self.regras = []

    def tell(self, sentence):
        sentence = sentence.strip()
        if ":-" in sentence:
            cabeca, corpo = sentence.split(":-", 1)
            self.regras.append((parse_atom(cabeca), [parse_atom(x) for x in dividir_argumentos(corpo)]))
        else:
            self.fatos.add(parse_atom(sentence))

    def _provar_atom(self, atom, subst, profundidade=0):
        if profundidade > 30:
            return []
        nome, termos = atom
        respostas = []
        # Fatos conhecidos
        for fnome, fargs in self.fatos:
            if nome == fnome:
                ns = unificar_argumentos(termos, fargs, subst)
                if ns is not None:
                    respostas.append(ns)
        # Regras Horn
        for (hnome, hargs), corpo in self.regras:
            if nome != hnome:
                continue
            ns = unificar_argumentos(termos, hargs, subst)
            if ns is None:
                continue
            estados = [ns]
            for cond in corpo:
                novos = []
                for est in estados:
                    novos.extend(self._provar_atom(cond, est, profundidade + 1))
                estados = novos
                if not estados:
                    break
            respostas.extend(estados)
        return respostas

    def ask(self, query):
        atom = parse_atom(query)
        respostas = self._provar_atom(atom, {})
        # Retorna substituições das variáveis consultadas; consulta sem variável vira bool.
        variaveis = [t for t in atom[1] if eh_variavel(t)]
        if not variaveis:
            return bool(respostas)
        saida = []
        vistos = set()
        for r in respostas:
            item = {v: r[v] for v in variaveis if v in r}
            chave = tuple(sorted(item.items()))
            if item and chave not in vistos:
                vistos.add(chave)
                saida.append(item)
        return saida

    def forward_chain(self):
        # Reaplica as regras até não surgirem fatos novos.
        alterou = True
        while alterou:
            alterou = False
            for (nome, termos), corpo in self.regras:
                estados = [{}]
                for cond in corpo:
                    novos = []
                    for est in estados:
                        novos.extend(self._provar_atom(cond, est))
                    estados = novos
                    if not estados:
                        break
                for est in estados:
                    args = tuple(valor_termo(t, est) for t in termos)
                    fato = (nome, args)
                    if fato not in self.fatos and all(not eh_variavel(x) for x in args):
                        self.fatos.add(fato)
                        alterou = True
        return self


## 3. Etapa 1 — mapeamento por metadados e lógica proposicional

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return texto.lower()

def simbolo_seguro(texto):
    s = re.sub(r"\W+", "_", normalizar(texto)).strip("_")
    if not s:
        s = "SemNome"
    if s[0].isdigit():
        s = "N_" + s
    return s

def mapear_metadados(raiz):
    registros = []
    for pasta_atual, dirs, arquivos in os.walk(raiz):
        dirs.sort()
        arquivos.sort()
        p = Path(pasta_atual)
        for d in dirs:
            caminho = p / d
            try:
                ano = time.localtime(caminho.stat().st_mtime).tm_year
            except OSError:
                ano = None
            registros.append({"tipo":"pasta", "nome":d, "caminho":str(caminho), "extensao":"", "ano_modificacao":ano, "pasta_pai":str(p)})
        for a in arquivos:
            caminho = p / a
            try:
                ano = time.localtime(caminho.stat().st_mtime).tm_year
            except OSError:
                ano = None
            registros.append({"tipo":"arquivo", "nome":a, "caminho":str(caminho), "extensao":caminho.suffix.lower(), "ano_modificacao":ano, "pasta_pai":str(p)})
    return pd.DataFrame(registros)

metadados = mapear_metadados(RAIZ)
display(metadados.head(20))
print(f"Total de pastas mapeadas: {(metadados.tipo == 'pasta').sum()}")
print(f"Total de arquivos mapeados: {(metadados.tipo == 'arquivo').sum()}")

,tipo,nome,caminho,extensao,ano_modificacao,pasta_pai
0,pasta,Fontes BD,/content/arquivo_professor/Fontes/Fontes BD,,2026,/content/arquivo_professor/Fontes
1,pasta,Fontes DMN,/content/arquivo_professor/Fontes/Fontes DMN,,2026,/content/arquivo_professor/Fontes
2,pasta,Fontes GD,/content/arquivo_professor/Fontes/Fontes GD,,2026,/content/arquivo_professor/Fontes
3,pasta,Fontes GTI,/content/arquivo_professor/Fontes/Fontes GTI,,2026,/content/arquivo_professor/Fontes
4,pasta,Fontes MD,/content/arquivo_professor/Fontes/Fontes MD,,2026,/content/arquivo_professor/Fontes
5,pasta,Fontes PAA,/content/arquivo_professor/Fontes/Fontes PAA,,2026,/content/arquivo_professor/Fontes
6,pasta,Fontes PCD,/content/arquivo_professor/Fontes/Fontes PCD,,2026,/content/arquivo_professor/Fontes
7,pasta,Fontes PDS,/content/arquivo_professor/Fontes/Fontes PDS,,2026,/content/arquivo_professor/Fontes
8,pasta,Fontes UDP,/content/arquivo_professor/Fontes/Fontes UDP,,2026,/content/arquivo_professor/Fontes
9,arquivo,Banco-de-Dados.pdf,/content/arquivo_professor/Fontes/Fontes BD/Ba...,.pdf,2026,/content/arquivo_professor/Fontes/Fontes BD


Total de pastas mapeadas: 10
Total de arquivos mapeados: 62


In [ ]:
kb_prop = KnowledgeBase()

# Fatos sobre metadados. Não abre o conteúdo dos arquivos.
for _, r in metadados.iterrows():
    caminho = r["caminho"].replace('"', "'")
    if r["tipo"] == "pasta":
        kb_prop.tell(f'Pasta("{caminho}")')
    else:
        kb_prop.tell(f'Arquivo("{caminho}")')
        ext = r["extensao"] or ".sem_extensao"
        kb_prop.tell(f'Extensao("{caminho}", "{ext}")')
    if pd.notna(r["ano_modificacao"]):
        kb_prop.tell(f'ModificadoEm("{caminho}", "{int(r["ano_modificacao"])}")')

    # Símbolos proposicionais descritivos adicionais
    if r["tipo"] == "arquivo" and r["extensao"] == ".pdf":
        kb_prop.tell(f'ExtensaoPDF("{caminho}")')
    if "plano" in normalizar(r["nome"]):
        kb_prop.tell(f'NomeRelacionadoPlanos("{caminho}")')

# Regras de negócio demonstrativas: PDF recente ou nome relacionado a planos é prioritário.
# O ano recente é relativo ao ano atual do sistema; altere o limiar se o professor especificar outro.
ANO_ATUAL = time.localtime().tm_year
ANO_LIMIAR = ANO_ATUAL - 2
for _, r in metadados[metadados.tipo == "arquivo"].iterrows():
    caminho = r["caminho"].replace('"', "'")
    if r["extensao"] == ".pdf" and pd.notna(r["ano_modificacao"]) and int(r["ano_modificacao"]) >= ANO_LIMIAR:
        kb_prop.tell(f'PDFRecente("{caminho}")')

kb_prop.tell('PrioridadeAlta(X) :- PDFRecente(X)')
kb_prop.tell('PrioridadeAlta(X) :- NomeRelacionadoPlanos(X)')
kb_prop.forward_chain()

prioritarios = kb_prop.ask("PrioridadeAlta(X)")
print("Consulta ASK(PrioridadeAlta(X)) — primeiros 20 resultados:")
for item in prioritarios[:20]:
    print(item)
print("Total de arquivos prioritários inferidos:", len(prioritarios))

# Exemplo de consulta proposicional para um arquivo real
arquivos_pdf = metadados[(metadados.tipo == "arquivo") & (metadados.extensao == ".pdf")]
if len(arquivos_pdf):
    exemplo = arquivos_pdf.iloc[0]["caminho"].replace('"', "'")
    print("ASK(ExtensaoPDF(caminho do primeiro PDF)) =", kb_prop.ask(f'ExtensaoPDF("{exemplo}")'))

Consulta ASK(PrioridadeAlta(X)) — primeiros 20 resultados:
{'X': '/content/arquivo_professor/Fontes/Fontes PCD/Guia Rápido_ Minimização de Lógica Digital.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes GD/Ferramentas gestao de dados/Oat+Gestão+de+dados.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes BD/Sistemas de Banco de Dados - Elmasri.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes PDS/apostila pds.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes GD/gestao-e-governança-de-dados.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes GD/Governança e boas práticas na Lei Geral de Proteção de Dados por meio da conformidade, da gestão de riscos e da accountability.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes DMN/Mobile_Development_Strategy_And_Code.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes PAA/Meta-Heurísticas em Pesquisa Operacional.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes PDS/apostila_pds.pdf'}
{'X': '/content/arquivo_professor/F

## 4. Etapa 2 — Lógica de Primeira Ordem (FOL)


In [ ]:
kb_fol = KnowledgeBase()

for _, r in metadados.iterrows():
    caminho = r["caminho"].replace('"', "'")
    pai = str(r["pasta_pai"]).replace('"', "'")
    if r["tipo"] == "pasta":
        kb_fol.tell(f'Pasta("{caminho}")')
        kb_fol.tell(f'Contem("{pai}", "{caminho}")')
    else:
        kb_fol.tell(f'Arquivo("{caminho}")')
        kb_fol.tell(f'Contem("{pai}", "{caminho}")')
        ext = (r["extensao"] or ".sem_extensao").replace('"', "'")
        kb_fol.tell(f'Extensao("{caminho}", "{ext}")')
        nome_norm = normalizar(r["nome"])
        for palavra, tema in [
            ("algoritmo", "algoritmos"), ("programacao", "programacao"),
            ("banco de dados", "banco_de_dados"), ("sistemas operacionais", "sistemas_operacionais"),
            ("inteligencia artificial", "inteligencia_artificial")
        ]:
            if palavra in nome_norm:
                kb_fol.tell(f'Tema("{caminho}", "{tema}")')

# Regra universal em forma de cláusula Horn:
# Todo PDF cujo nome indica tema acadêmico pode ser indicado para revisão.
# Para o laboratório, a regra é específica: arquivo com tema algoritmos e extensão PDF.
kb_fol.tell('RevisarParaProva(X) :- Arquivo(X), Extensao(X, ".pdf"), Tema(X, "algoritmos")')
kb_fol.forward_chain()

print("ASK(RevisarParaProva(X)) — arquivos encontrados:")
res_revisao = kb_fol.ask("RevisarParaProva(X)")
for resultado in res_revisao:
    print("X =", resultado["X"])

# Se não houver arquivos identificados pelo nome, mostre essa limitação explicitamente.
if not res_revisao:
    print("Nenhum PDF teve tema 'algoritmos' inferido apenas pelo nome. Isso não prova que não exista conteúdo sobre algoritmos; seria necessário analisar o conteúdo ou adicionar metadados confiáveis.")

print("\\nExemplo de relação Contem(PastaPai, Arquivo):")
exemplos_contidos = kb_fol.ask('Contem(X, Y)')
for x in exemplos_contidos[:10]:
    print(x)

ASK(RevisarParaProva(X)) — arquivos encontrados:
Nenhum PDF teve tema 'algoritmos' inferido apenas pelo nome. Isso não prova que não exista conteúdo sobre algoritmos; seria necessário analisar o conteúdo ou adicionar metadados confiáveis.
\nExemplo de relação Contem(PastaPai, Arquivo):
{'X': '/content/arquivo_professor/Fontes/Fontes DMN', 'Y': '/content/arquivo_professor/Fontes/Fontes DMN/Livro Desenvolvimento Mobile.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes UDP', 'Y': '/content/arquivo_professor/Fontes/Fontes UDP/Introdução e boas práticas em UX Design.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes GD/Ferramentas gestao de dados', 'Y': '/content/arquivo_professor/Fontes/Fontes GD/Ferramentas gestao de dados/Oat+Gestão+de+dados.pdf'}
{'X': '/content/arquivo_professor/Fontes/Fontes UDP', 'Y': '/content/arquivo_professor/Fontes/Fontes UDP/Conhecendo a Interface do Figma e as principais ferramentas de trabalho.pdf'}
{'X': '/content/arquivo_professor/Fontes', 'Y': '/cont

## 5. Etapa 3 — Busca comparativa e poda lógica



In [ ]:
# Comparação: execução somente do KB para evitar repetir BFS e A*
# A busca verifica o conteúdo real dos arquivos e para no primeiro resultado.

import time
import tracemalloc
import itertools
import heapq
import unicodedata
from pathlib import Path
import pandas as pd

try:
    from pypdf import PdfReader
except ImportError:
    !pip -q install pypdf
    from pypdf import PdfReader


def normalizar_busca(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    return texto.encode("ascii", "ignore").decode("ascii").lower()


EXTENSOES_TEXTO = {
    ".pdf", ".txt", ".md", ".csv",
    ".py", ".java", ".html", ".xml"
}


def arquivo_contem_alvo(caminho, alvo):
    """Procura o alvo no conteúdo real do arquivo."""
    caminho = Path(caminho)
    alvo_norm = normalizar_busca(alvo)

    if caminho.suffix.lower() == ".pdf":
        try:
            leitor = PdfReader(str(caminho))
            for pagina in leitor.pages:
                texto = pagina.extract_text() or ""
                if alvo_norm in normalizar_busca(texto):
                    return True
        except Exception:
            return False
        return False

    if caminho.suffix.lower() in EXTENSOES_TEXTO:
        try:
            texto = caminho.read_text(
                encoding="utf-8", errors="ignore"
            )
            return alvo_norm in normalizar_busca(texto)
        except Exception:
            return False

    return False


def medir_busca(metodo, alvo="algoritmo"):
    inicio = time.perf_counter()
    tracemalloc.start()

    pastas_abertas = 0
    arquivos_lidos = 0
    falsos_positivos = 0
    bytes_lidos = 0
    caminho_encontrado = None

    fila = []
    contador = itertools.count()
    heapq.heappush(
        fila, (0, next(contador), str(RAIZ), 0)
    )
    visitadas = set()

    while fila:
        prioridade, _, pasta, custo = heapq.heappop(fila)
        pasta_path = Path(pasta)

        if pasta in visitadas:
            continue
        visitadas.add(pasta)

        # Poda do KB: ignora somente pastas claramente inválidas.
        if metodo == "KB" and pasta_path != Path(RAIZ):
            nome_pasta = normalizar_busca(pasta_path.name)

            invalida = any(
                termo in nome_pasta
                for termo in [
                    "backup", "antigo",
                    "temporario", "temp"
                ]
            )

            if invalida:
                continue

        try:
            itens = sorted(
                pasta_path.iterdir(),
                key=lambda p: p.name.lower()
            )
        except (PermissionError, OSError, NotADirectoryError):
            continue

        pastas_abertas += 1

        subpastas = [p for p in itens if p.is_dir()]
        arquivos_locais = [p for p in itens if p.is_file()]

        for sub in subpastas:
            novo_custo = custo + 1

            if metodo == "A*":
                nome = normalizar_busca(sub.name)
                h = 0 if normalizar_busca(alvo) in nome else 1
                heapq.heappush(
                    fila,
                    (novo_custo + h, next(contador),
                     str(sub), novo_custo)
                )
            else:
                heapq.heappush(
                    fila,
                    (novo_custo, next(contador),
                     str(sub), novo_custo)
                )

        for arq in arquivos_locais:
            if arq.suffix.lower() not in EXTENSOES_TEXTO:
                continue

            try:
                tamanho = arq.stat().st_size
            except OSError:
                tamanho = 0

            arquivos_lidos += 1
            bytes_lidos += tamanho

            if arquivo_contem_alvo(arq, alvo):
                caminho_encontrado = str(arq)
                break

            falsos_positivos += 1

        if caminho_encontrado:
            break

    atual, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    tempo_ms = (time.perf_counter() - inicio) * 1000

    return {
        "Algoritmo": metodo,
        "Arquivo encontrado": caminho_encontrado or "Não encontrado",
        "Pastas abertas (nós expandidos)": pastas_abertas,
        "Arquivos lidos": arquivos_lidos,
        "Falsos positivos acessados": falsos_positivos,
        "Bytes lidos (tamanho em disco)": bytes_lidos,
        "Tempo (ms)": round(tempo_ms, 2),
        "Pico de memória (KB)": round(pico / 1024, 2)
    }


# Executa somente o KB
resultado_kb = medir_busca("KB", "algoritmo")

print("Resultado da busca KB:")
print(resultado_kb)

# Atualiza a tabela sem executar novamente BFS ou A*
if "tabela_resultados" in globals():
    tabela_atualizada = pd.concat(
        [
            tabela_resultados[
                tabela_resultados["Algoritmo"].isin(["BFS", "A*"])
            ],
            pd.DataFrame([resultado_kb])
        ],
        ignore_index=True
    )

    display(tabela_atualizada)
else:
    print(
        "Os resultados anteriores não estão disponíveis nesta sessão. "
        "O KB foi executado, mas BFS e A* não foram repetidos."
    )

Resultado da busca KB:
{'Algoritmo': 'KB', 'Arquivo encontrado': '/content/arquivo_professor/Fontes/Fontes BD/Sistemas de Banco de Dados - Elmasri.pdf', 'Pastas abertas (nós expandidos)': 2, 'Arquivos lidos': 5, 'Falsos positivos acessados': 4, 'Bytes lidos (tamanho em disco)': 31583154, 'Tempo (ms)': 151356.66, 'Pico de memória (KB)': 53806.91}


,Algoritmo,Arquivo encontrado,Pastas abertas (nós expandidos),Arquivos lidos,Falsos positivos acessados,Bytes lidos (tamanho em disco),Tempo (ms),Pico de memória (KB)
0,BFS,/content/arquivo_professor/Fontes/Fontes BD/Si...,2,5,4,31583154,157904.74,58469.77
1,A*,/content/arquivo_professor/Fontes/Fontes BD/Si...,2,5,4,31583154,156570.05,58559.80
2,KB,/content/arquivo_professor/Fontes/Fontes BD/Si...,2,5,4,31583154,151356.66,53806.91


## 6. Heurística lógica e interpretação da poda



In [ ]:
# Corrige os fatos de validação das pastas na base lógica

from pathlib import Path

# Usa os metadados já existentes, sem abrir os PDFs
pastas = metadados[metadados["tipo"] == "pasta"]

for pasta in pastas["caminho"]:
    p = str(pasta)
    nome = Path(p).name.lower()

    # Descarta somente pastas claramente temporárias ou de backup
    invalida = any(
        termo in nome
        for termo in ["backup", "antigo", "temporario", "temp"]
    )

    if not invalida:
        kb_logica.tell(f'PastaValida("{p}")')

print("Verificação das pastas:")
for pasta in pastas["caminho"].head(10):
    p = str(pasta)
    print(
        Path(p).name,
        "=>",
        kb_logica.ask(f'PastaValida("{p}")')
    )

Verificação das pastas:
Fontes BD => True
Fontes DMN => True
Fontes GD => True
Fontes GTI => True
Fontes MD => True
Fontes PAA => True
Fontes PCD => True
Fontes PDS => True
Fontes UDP => True
Ferramentas gestao de dados => True


In [ ]:
pdf = metadados[
    (metadados["tipo"] == "arquivo") &
    (metadados["extensao"] == ".pdf")
].iloc[0]["caminho"]

print("Arquivo:", pdf)
print("É PDF:", kb_prop.ask(f'ExtensaoPDF("{pdf}")'))

Arquivo: /content/arquivo_professor/Fontes/Fontes BD/Banco-de-Dados.pdf
É PDF: True


In [ ]:
from pathlib import Path

# Identifica o DataFrame de metadados já existente
if "metadados" in globals():
    df = metadados
elif "df_metadados" in globals():
    df = df_metadados
else:
    print("DataFrame de metadados não encontrado. Execute a célula que mapeia as pastas.")
    df = None

if df is not None:
    print("Pastas válidas:")

    if "tipo" in df.columns:
        pastas = df[df["tipo"].astype(str).str.lower() == "pasta"]
    else:
        pastas = df

    for pasta in pastas["caminho"].head(10):
        p = str(pasta).replace('"', "'")
        print(
            Path(p).name,
            "=>",
            kb_logica.ask(f'PastaValida("{p}")')
        )

Pastas válidas:
Fontes BD => True
Fontes DMN => True
Fontes GD => True
Fontes GTI => True
Fontes MD => True
Fontes PAA => True
Fontes PCD => True
Fontes PDS => True
Fontes UDP => True
Ferramentas gestao de dados => True
